In [1]:
import pandas as pd 
# 데이터 불러오기(행정동 데이터)
file_1 = pd.read_csv("서울시 상권분석서비스(점포-행정동2023).csv",encoding="cp949")
file_2 = pd.read_csv("서울시 상권분석서비스(점포-행정동2024).csv",encoding="cp949")
file_3 = pd.read_csv("서울시 상권분석서비스(점포-행정동2025).csv",encoding="cp949")
file_dong=pd.concat([file_1,file_2,file_3 ],ignore_index=True)
file_dong.head()

# 데이터 불러오기 (자치구 데이터)
gu=pd.read_csv("서울시 상권분석서비스(영역-자치구).csv",encoding="cp949")
gu_slim=gu[["자치구_코드","자치구_명"]]
gu_slim.columns


# 데이터 확인하기
print(file_dong.shape)
file_dong.info()
file_dong.isnull().sum() # 결측치 확인

# 데이터 전처리 
file_dong["전체_점포수_통합"]=file_dong["전체_점포_수"].fillna(file_dong["점포_수"])
file_dong["전체_점포수_통합"].isnull().sum()

file_dong.columns

file_dong_slim=file_dong[["기준_년분기_코드", "행정동_코드", "행정동_코드_명",
           "서비스_업종_코드_명", "전체_점포수_통합",
           "개업_율","개업_점포_수","폐업_률","폐업_점포_수"]] # 필요한 컬럼만 추출
file_dong_slim.columns

# 행정동 & 자치구 데이터 병합

# 숫자-> 문자열로 변환
file_dong_slim["행정동_코드_문자"] = file_dong_slim["행정동_코드"].astype(str) 
# 문자열에서 5글자만 자르기(자치구 코드와 행정동 코드는 앞 5글자가 동일하므로)
file_dong_slim["자치구_코드_문자"] = file_dong_slim["행정동_코드_문자"].str[:5]
# 다시 숫자로 변환(gu.pd와 타입 맞추기 위해)
file_dong_slim["자치구_코드"] = file_dong_slim["자치구_코드_문자"].astype(int)
# 자치구명 붙히기
file_dong_slim = pd.merge(file_dong_slim, gu_slim, on="자치구_코드",how="left")

file_dong_slim=file_dong_slim.drop(columns=["행정동_코드_문자","자치구_코드_문자"])
# 검증
print(file_dong_slim["자치구_명"].isnull().sum())   
file_dong_slim.head()

(458767, 14)
<class 'pandas.DataFrame'>
RangeIndex: 458767 entries, 0 to 458766
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   기준_년분기_코드    458767 non-null  int64  
 1   행정동_코드       458767 non-null  int64  
 2   행정동_코드_명     458767 non-null  str    
 3   서비스_업종_코드    458767 non-null  str    
 4   서비스_업종_코드_명  458767 non-null  str    
 5   점포_수         282236 non-null  float64
 6   유사_업종_점포_수   282236 non-null  float64
 7   개업_율         458767 non-null  int64  
 8   개업_점포_수      458767 non-null  int64  
 9   폐업_률         458767 non-null  int64  
 10  폐업_점포_수      458767 non-null  int64  
 11  프랜차이즈_점포_수   458767 non-null  int64  
 12  전체_점포_수      176531 non-null  float64
 13  일반_점포_수      176531 non-null  float64
dtypes: float64(4), int64(7), str(3)
memory usage: 49.0 MB
0


,기준_년분기_코드,행정동_코드,행정동_코드_명,서비스_업종_코드_명,전체_점포수_통합,개업_율,개업_점포_수,폐업_률,폐업_점포_수,자치구_코드,자치구_명
0,20231,11110515,청운효자동,한식음식점,74.0,4,3,4,3,11110,종로구
1,20231,11110515,청운효자동,중식음식점,4.0,0,0,0,0,11110,종로구
2,20231,11110515,청운효자동,일식음식점,17.0,0,0,0,0,11110,종로구
3,20231,11110515,청운효자동,양식음식점,48.0,2,1,4,2,11110,종로구
4,20231,11110515,청운효자동,제과점,23.0,0,0,0,0,11110,종로구


In [ ]:
)

In [2]:
#자치구+분기 단위로 묶어서 폐업률 추이 분석

gu_summary = (
    file_dong_slim.groupby(["자치구_명","기준_년분기_코드"])
).agg(
    전체_점포수=("전체_점포수_통합","sum"),
    개업_점포수=("개업_점포_수","sum"),
    폐업_점포수=("폐업_점포_수","sum")
).reset_index()

# 폐업률 & 개업률 계산

# 합산값 기준으로 비율 재계산
gu_summary["폐업률"] = gu_summary["폐업_점포수"] / gu_summary["전체_점포수"] * 100
gu_summary["개업률"] = gu_summary["개업_점포수"] / gu_summary["전체_점포수"] * 100

gu_summary["폐업률"] = gu_summary["폐업률"].round(2)
gu_summary["개업률"] = gu_summary["개업률"].round(2)

gu_summary["전체_점포수"] = gu_summary["전체_점포수"].astype(int)

print(gu_summary.shape)
gu_summary.head(10)

# 최신분기 기준 폐업률 랭킹
최신분기 = gu_summary["기준_년분기_코드"].max()
ranking = gu_summary[gu_summary["기준_년분기_코드"] == 최신분기].sort_values("폐업률", ascending=False)
ranking

(325, 7)


,자치구_명,기준_년분기_코드,전체_점포수,개업_점포수,폐업_점포수,폐업률,개업률
64,관악구,20261,21391,474,712,3.33,2.22
38,강북구,20261,13625,299,445,3.27,2.19
168,마포구,20261,32366,787,1030,3.18,2.43
285,은평구,20261,19610,381,604,3.08,1.94
51,강서구,20261,31341,712,962,3.07,2.27
324,중랑구,20261,17687,377,539,3.05,2.13
220,성북구,20261,18661,361,552,2.96,1.93
129,도봉구,20261,13101,218,386,2.95,1.66
246,양천구,20261,19939,357,580,2.91,1.79
25,강동구,20261,23949,483,683,2.85,2.02


In [24]:
# 1차 데이터 시각화

최신 = gu_summary[gu_summary["기준_년분기_코드"] == 최신분기].sort_values("폐업률", ascending=False).head(10)
최신.head(10)

,자치구_명,기준_년분기_코드,전체_점포수,개업_점포수,폐업_점포수,폐업률,개업률
64,관악구,20261,21391,474,712,3.33,2.22
38,강북구,20261,13625,299,445,3.27,2.19
168,마포구,20261,32366,787,1030,3.18,2.43
285,은평구,20261,19610,381,604,3.08,1.94
51,강서구,20261,31341,712,962,3.07,2.27
324,중랑구,20261,17687,377,539,3.05,2.13
220,성북구,20261,18661,361,552,2.96,1.93
129,도봉구,20261,13101,218,386,2.95,1.66
246,양천구,20261,19939,357,580,2.91,1.79
25,강동구,20261,23949,483,683,2.85,2.02


In [ ]:
apt = pd.read_csv("아파트매매거래.csv",encoding="utf-8-sig")
apt.head(10)

헤더행 = apt.iloc[0]
동호수_컬럼 = [col for col in apt.columns if 헤더행[col] == "동(호)수 (호수)"]
apt_slim = apt[["자치구별(2)"] + 동호수_컬럼].iloc[1:].reset_index(drop=True)
apt_slim.head()

In [16]:
gu_summary.head(10)

,자치구_명,기준_년분기_코드,전체_점포수,개업_점포수,폐업_점포수,폐업률,개업률
0,강남구,20231,59392,2218,1782,3.00,3.73
1,강남구,20232,59965,2168,1585,2.64,3.62
2,강남구,20233,60310,2024,1696,2.81,3.36
3,강남구,20234,60822,2134,1602,2.63,3.51
4,강남구,20241,60354,1546,2068,3.43,2.56
5,강남구,20242,60159,1521,1710,2.84,2.53
6,강남구,20243,59894,1467,1771,2.96,2.45
7,강남구,20244,59857,1421,1450,2.42,2.37
8,강남구,20251,63773,1088,1765,2.77,1.71
9,강남구,20252,63572,1297,1427,2.24,2.04


In [27]:
file_dong.shape

(458767, 15)